# Notebook 18 — Random Forest SQLi-Only Evaluation (EC2 CPU)

## Environment
- **Instance**: AWS EC2 (CPU only — no GPU required)
- **Framework**: scikit-learn + joblib

## Objective
Evaluate the Random Forest model (NB07) for SQL injection detection against the
**same SQLi-only subset of CSIC 2010** used in NB17, enabling a direct apples-to-apples
comparison between RF and MobileBERT.

## Key differences from earlier NB18 attempts
| Issue | Fix |
|---|---|
| `pickle.load()` fails (sklearn version mismatch 1.4.2 → 1.8.0) | Use `joblib.load()` instead |
| Only TF-IDF features passed to model | Combined matrix: `hstack([ngram, sym])` matching NB07 training |

## Feature matrix (must match NB07 training exactly)
```
X = hstack([tfidf_ngram_features, symbol_count_features])
```
Symbol features count occurrences of 17 special characters per query:
`'  "  ;  --  #  /*  */  *  +  |  (  )  >  <  \\  /  =`

## Dataset (identical to NB17)
- **Positive class**: 1,389 SQLi-only entries from CSIC 2010
- **Negative class**: 56,000 benign entries from CSIC 2010

## 0. Install Dependencies

In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'scikit-learn', 'pandas', 'numpy', 'scipy'])
print('Dependencies installed.')

Dependencies installed.


## 1. Imports & Paths

In [2]:
import os
import re
import time
import json
import joblib
import random
import warnings
import urllib.parse
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────
BASE         = os.path.expanduser('~/webattack-detector')
RESULTS_DIR  = f'{BASE}/notebooks/results/metrics'
MODELS_DIR   = f'{BASE}/notebooks/results/models'

ATTACK_FILE     = f'{BASE}/logs/csic2010/anomalousTrafficTest.txt'
BENIGN_TEST     = f'{BASE}/logs/csic2010/normalTrafficTest.txt'
BENIGN_TRAIN    = f'{BASE}/logs/csic2010/normalTrafficTraining.txt'
RF_MODEL_PATH   = f'{MODELS_DIR}/07_rf_model.pkl'
VECTORIZER_PATH = f'{MODELS_DIR}/07_vectorizer.pkl'

os.makedirs(RESULTS_DIR, exist_ok=True)
random.seed(42)

# ── Verify files ──────────────────────────────────────────────────
print()
all_ok = True
for label, path in [
    ('Attack file',  ATTACK_FILE),
    ('Benign test',  BENIGN_TEST),
    ('Benign train', BENIGN_TRAIN),
    ('RF model',     RF_MODEL_PATH),
    ('Vectorizer',   VECTORIZER_PATH),
]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) // 1024 if exists else 0
    status = '✅' if exists else '❌'
    print(f'  {status} {label:<14}: {size:>6} KB')
    if not exists:
        all_ok = False

print('\nAll files found. Ready.' if all_ok else '\n⚠️  Some files missing.')


  ✅ Attack file   :  15713 KB
  ✅ Benign test   :  20116 KB
  ✅ Benign train  :  20157 KB
  ✅ RF model      :   3189 KB
  ✅ Vectorizer    :    116 KB

All files found. Ready.


## 2. CSIC 2010 Parser

Identical to NB17 — fixed parser that correctly separates HTTP headers from POST body.

In [3]:
HTTP_METHODS = ('GET', 'POST', 'PUT', 'DELETE', 'HEAD', 'OPTIONS')

def parse_csic_file(filepath, label):
    entries = []
    with open(filepath, 'rb') as f:
        raw = f.read()

    if b'\r\n\r\n\r\n' in raw:
        request_sep = b'\r\n\r\n\r\n'
        header_sep  = b'\r\n\r\n'
        line_sep    = '\r\n'
    elif b'\r\n\r\n' in raw:
        request_sep = b'\r\n\r\n'
        header_sep  = None
        line_sep    = '\r\n'
    else:
        request_sep = b'\n\n\n'
        header_sep  = b'\n\n'
        line_sep    = '\n'

    def clean_body(body_text, ls):
        lines = body_text.split(ls)
        clean = []
        for line in lines:
            stripped = line.strip()
            if any(stripped.startswith(m + ' ') for m in HTTP_METHODS) \
                    and 'HTTP/' in stripped:
                break
            clean.append(line)
        return ls.join(clean).strip()

    def process_block(block_bytes):
        block = block_bytes.strip()
        if not block:
            return None
        if b'\r\n\r\n' in block:
            hdr_bytes, _, body_bytes = block.partition(b'\r\n\r\n')
            ls = '\r\n'
        elif b'\n\n' in block:
            hdr_bytes, _, body_bytes = block.partition(b'\n\n')
            ls = '\n'
        else:
            hdr_bytes  = block
            body_bytes = b''
            ls         = '\r\n'

        header_text = hdr_bytes.decode('latin-1', errors='ignore')
        body_raw    = body_bytes.decode('latin-1', errors='ignore').strip()
        body_text   = clean_body(body_raw, ls)

        lines = header_text.split(ls)
        first = lines[0].strip()
        parts = first.split(' ')

        if len(parts) < 2:
            return None
        if parts[0].upper() not in HTTP_METHODS:
            return None
        if 'HTTP/' not in first:
            return None

        try:
            parsed = urllib.parse.urlparse(parts[1])
        except Exception:
            return None

        post_params = {}
        if parts[0].upper() == 'POST' and body_text:
            try:
                first_body_line = body_text.split(ls)[0].strip()
                post_params = urllib.parse.parse_qs(
                    first_body_line, keep_blank_values=True
                )
            except Exception:
                pass

        return {
            'method':      parts[0].upper(),
            'url':         parts[1],
            'protocol':    parts[2] if len(parts) > 2 else 'HTTP/1.1',
            'request_line': first,
            'path':        parsed.path,
            'qs':          parsed.query,
            'body':        body_text,
            'post_params': post_params,
            'label':       label,
        }

    if request_sep == b'\r\n\r\n' and header_sep is None:
        chunks  = raw.split(b'\r\n\r\n')
        current = b''
        for chunk in chunks:
            chunk = chunk.strip()
            if not chunk:
                continue
            decoded    = chunk.decode('latin-1', errors='ignore')
            first_line = decoded.split('\r\n')[0].strip()
            fparts     = first_line.split(' ')
            if (len(fparts) >= 2
                    and fparts[0].upper() in HTTP_METHODS
                    and 'HTTP/' in first_line):
                if current:
                    entry = process_block(current)
                    if entry:
                        entries.append(entry)
                current = chunk
            else:
                current = current + b'\r\n\r\n' + chunk
        if current:
            entry = process_block(current)
            if entry:
                entries.append(entry)
    else:
        for raw_req in raw.split(request_sep):
            entry = process_block(raw_req)
            if entry:
                entries.append(entry)

    print(f'  {filepath.split("/")[-1]}: {len(entries):,} entries')
    return entries


print('Loading CSIC 2010...')
attack_entries = parse_csic_file(ATTACK_FILE,  label=1)
benign_test    = parse_csic_file(BENIGN_TEST,   label=0)
benign_train   = parse_csic_file(BENIGN_TRAIN,  label=0)
benign_entries = benign_test + benign_train

print(f'\nAttack : {len(attack_entries):,}')
print(f'Benign : {len(benign_entries):,}')

print('\nPOST body check (first 3 POST attacks):')
for e in [e for e in attack_entries if e['method'] == 'POST'][:3]:
    print(f'  Body: {urllib.parse.unquote(e["body"][:100])}')

Loading CSIC 2010...
  anomalousTrafficTest.txt: 15,088 entries
  normalTrafficTest.txt: 28,000 entries
  normalTrafficTraining.txt: 28,000 entries

Attack : 15,088
Benign : 56,000

POST body check (first 3 POST attacks):
  Body: id=2&nombre=Jam�n+Ib�rico&precio=85&cantidad=';+DROP+TABLE+usuarios;+SELECT+*+FROM+datos+W
  Body: id=2/&nombre=Jam�n+Ib�rico&precio=85&cantidad=49&B1=A�adir+al+carrito
  Body: modo=entrar&login=bob@<SCRipt>alert(Paros)</scrIPT>.parosproxy.org&pwd=84m3ri156&rem


## 3. Attack Type Classifier

Identical to NB17 — 21 SQL pattern families plus secondary classifier.

In [4]:
SQL_PATTERNS = [
    r"(?i)(union[\s\+%09%0a%0d]+select)",
    r"(?i)(select[\s\+%09]+.{0,40}[\s\+%09]+from)",
    r"(?i)(insert[\s\+%09]+into)",
    r"(?i)(update[\s\+%09]+\w+[\s\+%09]+set)",
    r"(?i)(delete[\s\+%09]+from)",
    r"(?i)(drop[\s\+%09]+(table|database|schema))",
    r"(?i)(alter[\s\+%09]+(table|database))",
    r"(?i)(create[\s\+%09]+(table|user|database))",
    r"(?i)(or[\s\+%09]+[\'\"]?1[\'\"]?[\s\+%09]*=[\s\+%09]*[\'\"]?1)",
    r"(?i)(and[\s\+%09]+[\'\"]?1[\'\"]?[\s\+%09]*=[\s\+%09]*[\'\"]?1)",
    r"(?i)(or[\s\+%09]+['\"][^'\"]{0,10}['\"][\s\+%09]*=[\s\+%09]*['\"][^'\"]{0,10}['\"])",
    r"(?i)(sleep[\s\+%09]*\()",
    r"(?i)(benchmark[\s\+%09]*\()",
    r"(?i)(waitfor[\s\+%09]+delay)",
    r"(?i)(information_schema)",
    r"(?i)(load_file[\s\+%09]*\()",
    r"(?i)(into[\s\+%09]+(outfile|dumpfile))",
    r"(?i)(pg_sleep[\s\+%09]*\()",
    r"(?i)(xp_cmdshell)",
    r"(?i)('[\s\+%09]*;[\s\+%09]*(select|insert|update|delete|drop))",
    r"(?i)(--[\s\+%09]*$|#[\s\+%09]*$|/\*.*?\*/)",
    r"(';|';--|' --)",
]

XSS_PATTERNS = [
    r"(?i)(<script[\s>])", r"(?i)(javascript[\s\+%09]*:)",
    r"(?i)(onerror[\s\+%09]*=)", r"(?i)(onload[\s\+%09]*=)",
    r"(?i)(alert[\s\+%09]*\()", r"(?i)(<iframe[\s>])",
    r"(?i)(document\.cookie)", r"(?i)(eval[\s\+%09]*\()",
    r"(?i)(%3cscript)", r"(?i)(vbscript[\s\+%09]*:)",
]
PATH_TRAVERSAL_PATTERNS = [
    r"(\.\./)|(\.\.\\)", r"(%2e%2e%2f|%2e%2e/|\.\.\.%2f)",
    r"(%252e%252e)", r"(etc/passwd|etc/shadow|win\.ini|boot\.ini)",
]
BUFFER_OVERFLOW_PATTERNS = [r"(A{50,}|%41{50,})", r"(.{200,})"]
SSI_PATTERNS = [r"(?i)(<!--\s*#\s*(include|exec|echo|printenv|set)\s)", r"(?i)(<!--#)"]
HEADER_INJECTION_PATTERNS = [
    r"(?i)(set-cookie\s*:)", r"(?i)(%0d%0a|%0a%0d).*:",
    r"(?i)(%0[aA]Set-cookie)", r"(?i)(%0[dD]%0[aA]Set-cookie)",
    r"(?i)(%3[fF]%0[dD]%0[aA])", r"(?i)(%0[aA][A-Za-z\-]+:)",
]
NULL_BYTE_PATTERNS   = [r"(%00|%2500|\x00)"]
CSTI_PATTERNS        = [r"(?i)(csrf|xsrf)", r"(?i)(__requestverificationtoken)"]
RECON_PATTERNS = [
    r"(?i)\.(bak|old|inc|backup|orig|tmp|swp|~)$",
    r"(?i)\.(bak|old|inc|backup|orig|tmp|swp|~)[/.]",
    r"(?i)(IISSamples|_cti_pvt|_vti_|webcart|scripts/tools|webapp/examples)",
    r"(?i)\d{10,}\.(java|old|bak|jsp|gif|jpg)",
    r"(?i)localhost:\d+\.(old|OLD|java|bak)$",
    r"(?i)\.(Old|OLD|BAK|Inc|INC|java|gif\.java)$",
    r"(?i)/servlet/",
]
PARAMETER_TAMPERING_PATTERNS = [
    r"(?i)([a-z]+A=)", r"(%252[Bb]|%252F|%253F)", r"(%2500)",
]
INJECTED_PARAM_PATTERNS  = [r"'INJECTED_PARAM", r"INJECTED_PARAM"]
REGEX_INJECTION_PATTERNS = [r"(\.\*\?|\.\+|\[\^|\(\?:)", r"(\.\*|\\\w|\(\.\))"]
VALUE_TAMPERING_PATTERNS = [
    r"(precio=\d+[%+/|?&])", r"(cantidad=\d+[%+/|?&])",
    r"(id=\d+[/|?&])", r"(B1=.*[/|?&][^&]*$)",
    r"(B2=.*[/|?&][^&]*$)", r"(\|$|^\|)",
]


def classify_attack(entry):
    url_decoded  = urllib.parse.unquote(entry.get('url', ''))
    qs_decoded   = urllib.parse.unquote(entry.get('qs', ''))
    body_decoded = urllib.parse.unquote(
        entry.get('body', '').split('\r\n')[0].split('\n')[0]
    )
    text = f"{url_decoded} {qs_decoded} {body_decoded}"
    types = []
    if any(re.search(p, text) for p in SQL_PATTERNS):               types.append('SQLi')
    if any(re.search(p, text) for p in XSS_PATTERNS):               types.append('XSS')
    if any(re.search(p, text) for p in SSI_PATTERNS):               types.append('SSI Injection')
    if any(re.search(p, text) for p in HEADER_INJECTION_PATTERNS):  types.append('Header Injection')
    if any(re.search(p, text) for p in NULL_BYTE_PATTERNS):         types.append('Null Byte')
    if any(re.search(p, text) for p in PATH_TRAVERSAL_PATTERNS):    types.append('Path Traversal')
    if any(re.search(p, text) for p in BUFFER_OVERFLOW_PATTERNS):   types.append('Buffer Overflow')
    if any(re.search(p, text) for p in CSTI_PATTERNS):              types.append('CSTI/XSRF')
    if any(re.search(p, text) for p in RECON_PATTERNS):             types.append('Reconnaissance')
    if any(re.search(p, text) for p in PARAMETER_TAMPERING_PATTERNS): types.append('Parameter Tampering')
    if any(re.search(p, text) for p in INJECTED_PARAM_PATTERNS):    types.append('Injected Param')
    if any(re.search(p, text) for p in REGEX_INJECTION_PATTERNS):   types.append('Regex Injection')
    if any(re.search(p, text) for p in VALUE_TAMPERING_PATTERNS):   types.append('Value Tampering')
    if entry.get('method') in ('PUT', 'DELETE') and '/tienda1/' in entry.get('url', ''):
        types.append('HTTP Method Abuse')
    return types if types else ['Unclassified']


def classify_unclassified(entry):
    url_decoded  = urllib.parse.unquote(entry.get('url', ''))
    qs_decoded   = urllib.parse.unquote(entry.get('qs', ''))
    body_decoded = urllib.parse.unquote(
        entry.get('body', '').split('\r\n')[0].split('\n')[0]
    )
    text = f"{url_decoded} {qs_decoded} {body_decoded}"
    if re.search(r"(?i)\.(bak|old|inc|backup|orig|tmp|swp|~|Old|OLD|BAK|Inc|INC|java)(\b|$|/|\?)", url_decoded) or \
       re.search(r"(?i)(servlet|showcfg|drvrs|IISSamples|_cti|asf-logo|\d{10,})", url_decoded):
        return ['Reconnaissance']
    if entry.get('method') in ('PUT', 'DELETE', 'HEAD', 'OPTIONS'):
        return ['HTTP Method Abuse']
    if re.search(r"(\.\*\?|\.\+|\[\^|\\[dDwWsS]|\(\?)", text):
        return ['Regex Injection']
    if re.search(r"(=[^&]*[|@?%+][^&]*(&|$))", qs_decoded + body_decoded):
        return ['Value Tampering']
    if re.search(r"(=[^&]*[/][^&]*(&|$))", qs_decoded + body_decoded):
        return ['Value Tampering']
    if entry.get('method') == 'POST':
        return ['Parameter Tampering']
    return ['Reconnaissance']


print('Classifying attack types...')
for e in attack_entries:
    e['attack_types'] = classify_attack(e)
resolved = sum(1 for e in attack_entries if e['attack_types'] == ['Unclassified'])
for e in attack_entries:
    if e['attack_types'] == ['Unclassified']:
        e['attack_types'] = classify_unclassified(e)

total          = len(attack_entries)
primary_counts = Counter(e['attack_types'][0] for e in attack_entries)
unclassified   = primary_counts.get('Unclassified', 0)

print(f'\nTotal attack entries  : {total:,}')
print(f'Resolved unclassified : {resolved:,}')
print(f'Remaining unclassified: {unclassified}')
print()
print('─' * 58)
print(f'{"Attack Type":<36} {"Count":>7}  {"% of Total":>10}')
print('─' * 58)
for atype, count in primary_counts.most_common():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'{atype:<36} {count:>7,}  {pct:>9.1f}%  {bar}')
print('─' * 58)

Classifying attack types...

Total attack entries  : 15,088
Resolved unclassified : 4,093
Remaining unclassified: 0

──────────────────────────────────────────────────────────
Attack Type                            Count  % of Total
──────────────────────────────────────────────────────────
Value Tampering                        3,257       21.6%  ██████████
Reconnaissance                         3,166       21.0%  ██████████
Buffer Overflow                        2,685       17.8%  ████████
Parameter Tampering                    2,236       14.8%  ███████
SQLi                                   1,389        9.2%  ████
XSS                                      811        5.4%  ██
Header Injection                         653        4.3%  ██
SSI Injection                            510        3.4%  █
Null Byte                                121        0.8%  
Injected Param                           115        0.8%  
HTTP Method Abuse                         88        0.6%  
Regex Injection

## 4. SQLi Subset Extraction

In [5]:
sqli_entries = [e for e in attack_entries if 'SQLi' in e['attack_types']]
non_sqli     = [e for e in attack_entries if 'SQLi' not in e['attack_types']]

print('=' * 60)
print('SQLi SUBSET SUMMARY')
print('=' * 60)
print(f'Total CSIC attack entries  : {len(attack_entries):,}')
print(f'SQLi entries (positive)    : {len(sqli_entries):,}  '
      f'({len(sqli_entries)/len(attack_entries)*100:.1f}% of attacks)')
print(f'Non-SQLi attacks (excluded): {len(non_sqli):,}')
print(f'Benign entries (negative)  : {len(benign_entries):,}')
print()

false_sqli = []
for e in sqli_entries:
    text = (
        urllib.parse.unquote(e.get('url', '')) + ' ' +
        urllib.parse.unquote(e.get('qs', '')) + ' ' +
        urllib.parse.unquote(e.get('body', '').split('\r\n')[0].split('\n')[0])
    )
    if not any(re.search(p, text) for p in SQL_PATTERNS):
        false_sqli.append(e)

status = '✅' if len(false_sqli) == 0 else '❌'
print('─' * 60)
print('VERIFICATION')
print('─' * 60)
print(f'{status} Pattern confirmation: {len(sqli_entries)-len(false_sqli):,} / '
      f'{len(sqli_entries):,} confirmed by SQL pattern')

method_counts = Counter(e['method'] for e in sqli_entries)
print()
print('SQLi by HTTP method:')
for method, count in method_counts.most_common():
    pct = count / len(sqli_entries) * 100
    print(f'  {method:<8} {count:>5,}  ({pct:.1f}%)')

SQLi SUBSET SUMMARY
Total CSIC attack entries  : 15,088
SQLi entries (positive)    : 1,389  (9.2% of attacks)
Non-SQLi attacks (excluded): 13,699
Benign entries (negative)  : 56,000

────────────────────────────────────────────────────────────
VERIFICATION
────────────────────────────────────────────────────────────
✅ Pattern confirmation: 1,389 / 1,389 confirmed by SQL pattern

SQLi by HTTP method:
  POST       987  (71.1%)
  GET        402  (28.9%)


## 5. Load RF Model & Helpers

**Fix 1**: `joblib.load()` instead of `pickle.load()` — handles sklearn version mismatch.

**Fix 2**: Feature matrix `hstack([ngram, sym])` matches NB07 training exactly.

In [6]:
# ── Load via joblib (handles sklearn version differences) ─────────
print('Loading RF model and vectorizer...')
rf_model   = joblib.load(RF_MODEL_PATH)
vectorizer = joblib.load(VECTORIZER_PATH)

print(f'  ✅ RF model     : {RF_MODEL_PATH.split("/")[-1]}')
print(f'  ✅ Vectorizer   : {VECTORIZER_PATH.split("/")[-1]}')
print(f'  Estimators      : {rf_model.n_estimators}')
print(f'  Features        : {rf_model.n_features_in_:,}')
print(f'  Classes         : {rf_model.classes_}')
print(f'  Vocab size      : {len(vectorizer.vocabulary_):,}')


# ── Symbol feature matrix — must match NB07 training ─────────────
SYMBOLS = ["'", '"', ";", "--", "#", "/*", "*/", "*", "+",
           "|", "(", ")", ">", "<", "\\", "/", "="]

def build_symbol_matrix(queries):
    """
    Count occurrences of each special character per query.
    Returns sparse matrix of shape (n_queries, 17).
    Matches NB07 build_symbol_matrix exactly.
    """
    rows = []
    for q in queries:
        rows.append([str(q).count(sym) for sym in SYMBOLS])
    return csr_matrix(np.array(rows, dtype=float))


# ── Sanity check — confirms feature dimensions match model ────────
print('\nSanity check (3 test queries):')
test_queries = [
    "'; DROP TABLE usuarios; SELECT * FROM datos WHERE nombre LIKE '%",
    "Jamón Ibérico",
    "2",
]
ngram_test = vectorizer.transform(test_queries)
sym_test   = build_symbol_matrix(test_queries)
X_test     = hstack([ngram_test, sym_test])
probs_test = rf_model.predict_proba(X_test)[:, 1]
for q, p in zip(test_queries, probs_test):
    print(f'  {p:.4f}  {q[:60]}')
print(f'  Feature dims: {X_test.shape[1]:,} '
      f'(n-gram: {ngram_test.shape[1]:,} + symbol: {sym_test.shape[1]})')


# ── Extraction strategies (identical to NB17) ─────────────────────
def extract_request_line(entry):
    try:
        url = urllib.parse.unquote(entry['url']).strip()
        return f'{entry["method"]} {url} {entry.get("protocol", "HTTP/1.1")}' if url else None
    except Exception:
        return None

def extract_full_url(entry):
    try:
        return urllib.parse.unquote(entry['url']).strip() or None
    except Exception:
        return None

def extract_query_values(entry):
    """Query string values for GET; POST body values as fallback."""
    try:
        qs = entry.get('qs', '')
        if qs:
            params = urllib.parse.parse_qs(qs, keep_blank_values=False)
            values = [
                urllib.parse.unquote(v).strip()
                for vlist in params.values()
                for v in vlist
                if urllib.parse.unquote(v).strip()
            ]
            if values:
                return ' '.join(values)
        body = entry.get('body', '').split('\r\n')[0].split('\n')[0].strip()
        if body:
            params = urllib.parse.parse_qs(body, keep_blank_values=False)
            values = [
                urllib.parse.unquote(v).strip()
                for vlist in params.values()
                for v in vlist
                if urllib.parse.unquote(v).strip()
            ]
            if values:
                return ' '.join(values)
        return None
    except Exception:
        return None


# ── RF scoring — combined feature matrix ──────────────────────────
def score_rf(entries, extract_fn, model, vec, label=''):
    texts, indices = [], []
    for i, e in enumerate(entries):
        text = extract_fn(e)
        if text and text.strip():
            texts.append(text)
            indices.append(i)

    print(f'  {label} scoreable: {len(texts):,} / {len(entries):,}')
    if not texts:
        return [0.0] * len(entries)

    t0    = time.perf_counter()
    ngram = vec.transform(texts)
    sym   = build_symbol_matrix(texts)
    X     = hstack([ngram, sym])
    probs = model.predict_proba(X)[:, 1]
    elapsed = time.perf_counter() - t0
    print(f'  Scored {len(texts):,} in {elapsed:.2f}s ({len(texts)/elapsed:.0f}/sec)')

    result = [0.0] * len(entries)
    for idx, prob in zip(indices, probs):
        result[idx] = float(prob)
    return result


# ── Threshold scan ────────────────────────────────────────────────
def threshold_scan(label, probs_a, probs_b):
    print(f'\n{"="*70}')
    print(f'RF | {label}')
    print(f'{"="*70}')
    print(f'{"Threshold":>12} {"Recall":>8} {"FP/10k":>8} '
          f'{"Precision":>10} {"Detected":>10} {"F1":>8}')
    print('-' * 65)
    best = None
    for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
        detected  = sum(1 for s in probs_a if s >= t)
        fp        = sum(1 for s in probs_b if s >= t)
        recall    = detected / len(probs_a) if probs_a else 0
        fp_10k    = (fp / len(probs_b)) * 10000 if probs_b else 0
        precision = detected / (detected + fp) if (detected + fp) > 0 else 0
        f1        = (2 * precision * recall / (precision + recall)) \
                    if (precision + recall) > 0 else 0
        marker    = ' ←' if (best is None or f1 > best['f1']) else ''
        print(f'{t:>12.2f} {recall:>8.4f} {fp_10k:>8.2f} '
              f'{precision:>10.4f} {detected:>10,} {f1:>8.4f}{marker}')
        if best is None or f1 > best['f1']:
            best = {'threshold': t, 'recall': recall, 'fp_10k': fp_10k,
                    'precision': precision, 'f1': f1}
    print(f'\n  Best F1 @ threshold {best["threshold"]}: '
          f'recall={best["recall"]:.4f}, fp_10k={best["fp_10k"]:.2f}, '
          f'f1={best["f1"]:.4f}')
    return best


def best_threshold_fine(probs_a, probs_b):
    best = None
    for t in np.arange(0.3, 1.0, 0.01):
        detected  = sum(1 for s in probs_a if s >= t)
        fp        = sum(1 for s in probs_b if s >= t)
        recall    = detected / len(probs_a) if probs_a else 0
        precision = detected / (detected + fp) if (detected + fp) > 0 else 0
        f1        = (2 * precision * recall / (precision + recall)) \
                    if (precision + recall) > 0 else 0
        fp_10k    = (fp / len(probs_b)) * 10000 if probs_b else 0
        if best is None or f1 > best['f1']:
            best = {'t': round(float(t), 2), 'recall': recall,
                    'fp_10k': fp_10k, 'precision': precision, 'f1': f1}
    return best


print('\nAll helpers defined. Ready to evaluate.')

Loading RF model and vectorizer...
  ✅ RF model     : 07_rf_model.pkl
  ✅ Vectorizer   : 07_vectorizer.pkl
  Estimators      : 100
  Features        : 8,824
  Classes         : [0 1]
  Vocab size      : 8,807

Sanity check (3 test queries):
  0.9400  '; DROP TABLE usuarios; SELECT * FROM datos WHERE nombre LIK
  0.0800  Jamón Ibérico
  0.2200  2
  Feature dims: 8,824 (n-gram: 8,807 + symbol: 17)

All helpers defined. Ready to evaluate.


## 6. RUN 1 — Strategy 1: Full Request Line

**Input**: `GET /shop?id=1' UNION SELECT username,password FROM users-- HTTP/1.1`

Out-of-distribution for the RF (trained on query values only) — shows generalisation ability.

In [7]:
print('Scoring — Strategy 1: Full Request Line')
print('─' * 60)
t0 = time.perf_counter()
probs_a_s1 = score_rf(sqli_entries,   extract_request_line, rf_model, vectorizer, 'SQLi')
probs_b_s1 = score_rf(benign_entries, extract_request_line, rf_model, vectorizer, 'Benign')
print(f'\nTotal time: {time.perf_counter()-t0:.2f}s')

a, b = np.array(probs_a_s1), np.array(probs_b_s1)
print(f'\nScore distributions:')
print(f'  SQLi   — mean: {a.mean():.3f}  min: {a.min():.3f}  max: {a.max():.3f}')
print(f'  Benign — mean: {b.mean():.3f}  min: {b.min():.3f}  max: {b.max():.3f}')

best_s1 = threshold_scan('Full Request Line', probs_a_s1, probs_b_s1)
with open(f'{RESULTS_DIR}/18_probs_s1.json', 'w') as f:
    json.dump({'sqli': probs_a_s1, 'benign': probs_b_s1}, f)
print('\nProbabilities saved.')

Scoring — Strategy 1: Full Request Line
────────────────────────────────────────────────────────────
  SQLi scoreable: 1,389 / 1,389
  Scored 1,389 in 0.21s (6716/sec)
  Benign scoreable: 56,000 / 56,000
  Scored 56,000 in 6.05s (9263/sec)

Total time: 6.35s

Score distributions:
  SQLi   — mean: 0.667  min: 0.550  max: 0.890
  Benign — mean: 0.627  min: 0.550  max: 0.850

RF | Full Request Line
   Threshold   Recall   FP/10k  Precision   Detected       F1
-----------------------------------------------------------------
        0.30   1.0000 10000.00     0.0242      1,389   0.0473 ←
        0.40   1.0000 10000.00     0.0242      1,389   0.0473
        0.50   1.0000 10000.00     0.0242      1,389   0.0473
        0.60   0.8099  8928.39     0.0220      1,125   0.0428
        0.70   0.2858   853.93     0.0767        397   0.1209 ←
        0.80   0.1735    16.43     0.7237        241   0.2799 ←
        0.90   0.0000     0.00     0.0000          0   0.0000
        0.95   0.0000     0.00   

## 7. RUN 2 — Strategy 2: Full URL

**Input**: `/shop?id=1' UNION SELECT username,password FROM users--`

In [8]:
print('Scoring — Strategy 2: Full URL')
print('─' * 60)
t0 = time.perf_counter()
probs_a_s2 = score_rf(sqli_entries,   extract_full_url, rf_model, vectorizer, 'SQLi')
probs_b_s2 = score_rf(benign_entries, extract_full_url, rf_model, vectorizer, 'Benign')
print(f'\nTotal time: {time.perf_counter()-t0:.2f}s')

a, b = np.array(probs_a_s2), np.array(probs_b_s2)
print(f'\nScore distributions:')
print(f'  SQLi   — mean: {a.mean():.3f}  min: {a.min():.3f}  max: {a.max():.3f}')
print(f'  Benign — mean: {b.mean():.3f}  min: {b.min():.3f}  max: {b.max():.3f}')

best_s2 = threshold_scan('Full URL', probs_a_s2, probs_b_s2)
with open(f'{RESULTS_DIR}/18_probs_s2.json', 'w') as f:
    json.dump({'sqli': probs_a_s2, 'benign': probs_b_s2}, f)
print('\nProbabilities saved.')

Scoring — Strategy 2: Full URL
────────────────────────────────────────────────────────────
  SQLi scoreable: 1,389 / 1,389
  Scored 1,389 in 0.18s (7542/sec)
  Benign scoreable: 56,000 / 56,000
  Scored 56,000 in 5.22s (10723/sec)

Total time: 5.49s

Score distributions:
  SQLi   — mean: 0.656  min: 0.560  max: 0.880
  Benign — mean: 0.631  min: 0.560  max: 0.850

RF | Full URL
   Threshold   Recall   FP/10k  Precision   Detected       F1
-----------------------------------------------------------------
        0.30   1.0000 10000.00     0.0242      1,389   0.0473 ←
        0.40   1.0000 10000.00     0.0242      1,389   0.0473
        0.50   1.0000 10000.00     0.0242      1,389   0.0473
        0.60   0.6508  8571.07     0.0185        904   0.0360
        0.70   0.2858   815.54     0.0800        397   0.1250 ←
        0.80   0.1577    13.39     0.7449        219   0.2602 ←
        0.90   0.0000     0.00     0.0000          0   0.0000
        0.95   0.0000     0.00     0.0000         

## 8. RUN 3 — Strategy 3: Query Values Only (Way 3)

**Input**: `1' UNION SELECT username,password FROM users--`

This is the strategy the RF was trained on in NB07 — expected strongest result.

In [9]:
print('Scoring — Strategy 3: Query Values Only (Way 3)')
print('─' * 60)
t0 = time.perf_counter()
probs_a_s3 = score_rf(sqli_entries,   extract_query_values, rf_model, vectorizer, 'SQLi')
probs_b_s3 = score_rf(benign_entries, extract_query_values, rf_model, vectorizer, 'Benign')
print(f'\nTotal time: {time.perf_counter()-t0:.2f}s')

a, b = np.array(probs_a_s3), np.array(probs_b_s3)
print(f'\nScore distributions:')
print(f'  SQLi   — mean: {a.mean():.3f}  min: {a.min():.3f}  max: {a.max():.3f}')
print(f'  Benign — mean: {b.mean():.3f}  min: {b.min():.3f}  max: {b.max():.3f}')

best_s3 = threshold_scan('Query Values (Way 3)', probs_a_s3, probs_b_s3)
with open(f'{RESULTS_DIR}/18_probs_s3.json', 'w') as f:
    json.dump({'sqli': probs_a_s3, 'benign': probs_b_s3}, f)
print('\nProbabilities saved.')

Scoring — Strategy 3: Query Values Only (Way 3)
────────────────────────────────────────────────────────────
  SQLi scoreable: 1,389 / 1,389
  Scored 1,389 in 0.19s (7354/sec)
  Benign scoreable: 26,000 / 56,000
  Scored 26,000 in 2.44s (10640/sec)

Total time: 3.14s

Score distributions:
  SQLi   — mean: 0.888  min: 0.610  max: 1.000
  Benign — mean: 0.224  min: 0.000  max: 0.910

RF | Query Values (Way 3)
   Threshold   Recall   FP/10k  Precision   Detected       F1
-----------------------------------------------------------------
        0.30   1.0000  3959.11     0.0590      1,389   0.1113 ←
        0.40   1.0000  2912.14     0.0785      1,389   0.1456 ←
        0.50   1.0000  2225.18     0.1003      1,389   0.1823 ←
        0.60   1.0000  1497.14     0.1421      1,389   0.2489 ←
        0.70   0.9921   936.07     0.2082      1,378   0.3441 ←
        0.80   0.9143   118.57     0.6567      1,270   0.7644 ←
        0.90   0.4564     0.36     0.9969        634   0.6262
        0.95   

## 9. RF Results Summary

In [10]:
def load_probs(path):
    with open(path) as f:
        d = json.load(f)
    return d['sqli'], d['benign']

if 'probs_a_s1' not in dir(): probs_a_s1, probs_b_s1 = load_probs(f'{RESULTS_DIR}/18_probs_s1.json'); print('Loaded S1')
if 'probs_a_s2' not in dir(): probs_a_s2, probs_b_s2 = load_probs(f'{RESULTS_DIR}/18_probs_s2.json'); print('Loaded S2')
if 'probs_a_s3' not in dir(): probs_a_s3, probs_b_s3 = load_probs(f'{RESULTS_DIR}/18_probs_s3.json'); print('Loaded S3')

b1 = best_threshold_fine(probs_a_s1, probs_b_s1)
b2 = best_threshold_fine(probs_a_s2, probs_b_s2)
b3 = best_threshold_fine(probs_a_s3, probs_b_s3)

W = 100
print('=' * W)
print('RF RESULTS — SQLi-only subset of CSIC 2010')
print('1,389 SQLi attacks vs 56,000 benign | Best F1 threshold per row (step=0.01)')
print('=' * W)
print(f'{"Model":<8} {"Strategy":<30} {"Thresh":>7} '
      f'{"Recall":>8} {"FP/10k":>8} {"Precision":>10} {"F1":>8}')
print('-' * W)
for strat, b in [
    ('Full Request Line', b1),
    ('Full URL',          b2),
    ('Way 3',             b3),
]:
    print(f'{"RF":<8} {strat:<30} {b["t"]:>7.2f} '
          f'{b["recall"]:>8.4f} {b["fp_10k"]:>8.2f} '
          f'{b["precision"]:>10.4f} {b["f1"]:>8.4f}')
print('=' * W)

summary = [
    {'model': 'RF', 'strategy': 'Full Request Line',
     'threshold': b1['t'], 'recall': round(b1['recall'],4),
     'fp_10k': round(b1['fp_10k'],4), 'precision': round(b1['precision'],4),
     'f1': round(b1['f1'],4), 'notebook': 'NB18'},
    {'model': 'RF', 'strategy': 'Full URL',
     'threshold': b2['t'], 'recall': round(b2['recall'],4),
     'fp_10k': round(b2['fp_10k'],4), 'precision': round(b2['precision'],4),
     'f1': round(b2['f1'],4), 'notebook': 'NB18'},
    {'model': 'RF', 'strategy': 'Way 3',
     'threshold': b3['t'], 'recall': round(b3['recall'],4),
     'fp_10k': round(b3['fp_10k'],4), 'precision': round(b3['precision'],4),
     'f1': round(b3['f1'],4), 'notebook': 'NB18'},
]
pd.DataFrame(summary).to_csv(f'{RESULTS_DIR}/18_rf_sqli_only_summary.csv', index=False)
with open(f'{RESULTS_DIR}/18_rf_sqli_only_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: {RESULTS_DIR}/18_rf_sqli_only_summary.csv')
print(f'Saved: {RESULTS_DIR}/18_rf_sqli_only_summary.json')

RF RESULTS — SQLi-only subset of CSIC 2010
1,389 SQLi attacks vs 56,000 benign | Best F1 threshold per row (step=0.01)
Model    Strategy                        Thresh   Recall   FP/10k  Precision       F1
----------------------------------------------------------------------------------------------------
RF       Full Request Line                 0.77   0.2181    55.36     0.4943   0.3027
RF       Full URL                          0.76   0.2261    85.18     0.3970   0.2881
RF       Way 3                             0.82   0.8265    33.57     0.8593   0.8426

Saved: /home/ubuntu/webattack-detector/notebooks/results/metrics/18_rf_sqli_only_summary.csv
Saved: /home/ubuntu/webattack-detector/notebooks/results/metrics/18_rf_sqli_only_summary.json


## 10. Cost Analysis

Same 10 access log entries as NB17 — enables direct comparison.

**Important note on latency measurement**: at batch_size=1, the measured latency (~23ms) is
dominated by TF-IDF vectoriser overhead for a single string, not model inference.
The throughput numbers at batch_size=100 and 1000 are the representative production metric
and show RF's true advantage over MobileBERT.

Cost is shown for both batch=1 (strict real-time) and batch=100 (light batching) scenarios.

In [11]:
ACCESS_LOG_ENTRIES = [
    '192.168.1.1 - - [25/Jun/2024:10:01:01 +0000] "GET /shop/product?id=1\'OR\'1\'=\'1&Submit=Submit HTTP/1.1" 200 1234 "-" "Mozilla/5.0"',
    '192.168.1.2 - - [25/Jun/2024:10:01:02 +0000] "GET /search?q=test\'UNION+SELECT+username,password+FROM+users-- HTTP/1.1" 200 5678 "-" "Mozilla/5.0"',
    '192.168.1.3 - - [25/Jun/2024:10:01:03 +0000] "GET /item?id=1\';+waitfor+delay+\'0:0:5\';-- HTTP/1.1" 200 2345 "-" "Mozilla/5.0"',
    '192.168.1.4 - - [25/Jun/2024:10:01:04 +0000] "GET /user?login=admin\';+DROP+TABLE+users;-- HTTP/1.1" 200 3456 "-" "Mozilla/5.0"',
    '192.168.1.5 - - [25/Jun/2024:10:01:05 +0000] "GET /filter?cat=shoes\'AND+1=CAST((SELECT+pg_sleep(5))+AS+int)-- HTTP/1.1" 200 4567 "-" "Mozilla/5.0"',
    '192.168.1.6 - - [25/Jun/2024:10:01:06 +0000] "GET /auth?user=admin\'OR\'a\'=\'a HTTP/1.1" 200 5678 "-" "Mozilla/5.0"',
    '192.168.1.7 - - [25/Jun/2024:10:01:07 +0000] "GET /api?q=1+AND+1=1+UNION+SELECT+table_name+FROM+information_schema.tables-- HTTP/1.1" 200 6789 "-" "Mozilla/5.0"',
    '192.168.1.8 - - [25/Jun/2024:10:01:08 +0000] "GET /view?id=1%27%20OR%20%271%27%3D%271 HTTP/1.1" 200 7890 "-" "Mozilla/5.0"',
    '192.168.1.9 - - [25/Jun/2024:10:01:09 +0000] "GET /page?id=1/**/OR/**/1=1-- HTTP/1.1" 200 8901 "-" "Mozilla/5.0"',
    '192.168.1.10 - - [25/Jun/2024:10:01:10 +0000] "GET /sort?by=name\'AND+BENCHMARK(5000000,MD5(1))-- HTTP/1.1" 200 9012 "-" "Mozilla/5.0"',
]

def parse_access_log_entry(log_line):
    try:
        parts    = log_line.split('"')
        request  = parts[1]
        url_part = request.split(' ')[1]
        parsed   = urllib.parse.urlparse(url_part)
        qs       = parsed.query
        if not qs:
            return urllib.parse.unquote(url_part)
        params = urllib.parse.parse_qs(qs, keep_blank_values=False)
        values = [
            urllib.parse.unquote(v).strip()
            for vlist in params.values()
            for v in vlist
            if urllib.parse.unquote(v).strip()
        ]
        return ' '.join(values) if values else urllib.parse.unquote(url_part)
    except Exception:
        return None

parsed_queries = [parse_access_log_entry(l) for l in ACCESS_LOG_ENTRIES]
valid_queries  = [q for q in parsed_queries if q]

print('Parsed access log entries:')
print('─' * 70)
for i, q in enumerate(parsed_queries, 1):
    print(f'[{i:>2}] {q[:80] if q else "FAILED"}')
print(f'\nSuccessfully parsed: {len(valid_queries)}/{len(ACCESS_LOG_ENTRIES)}')


# ── Single request latency ────────────────────────────────────────
# Note: at batch_size=1, latency is dominated by TF-IDF single-item
# transform overhead (~23ms), not model inference. Throughput numbers
# at larger batch sizes are the representative production metric.
N_REPEATS = 100
print(f'\nMeasuring single-request latency ({N_REPEATS} repeats per query)...')

latencies = []
for query in valid_queries:
    for _ in range(N_REPEATS):
        t0    = time.perf_counter()
        ngram = vectorizer.transform([query])
        sym   = build_symbol_matrix([query])
        prob  = rf_model.predict_proba(hstack([ngram, sym]))[0, 1]
        t1    = time.perf_counter()
        latencies.append((t1 - t0) * 1000)

lat = np.array(latencies)
print(f'\n{"─" * 65}')
print('SINGLE REQUEST LATENCY (ms) — CPU, 1 request at a time')
print(f'{"─" * 65}')
print(f'{"Model":<35} {"Mean":>8} {"Std":>8} {"P50":>8} {"P95":>8} {"P99":>8}')
print(f'{"─" * 65}')
print(f'{"RF (TF-IDF + Random Forest)":<35} '
      f'{np.mean(lat):>8.3f} {np.std(lat):>8.3f} '
      f'{np.percentile(lat,50):>8.3f} '
      f'{np.percentile(lat,95):>8.3f} '
      f'{np.percentile(lat,99):>8.3f}')
print(f'{"─" * 65}')
print('  ⚠ ~23ms dominated by TF-IDF single-item transform overhead.')
print('  See throughput below for representative production numbers.')


# ── Throughput ────────────────────────────────────────────────────
BATCH_SIZES = (1, 10, 100, 1000)
print(f'\nMeasuring throughput at batch sizes {BATCH_SIZES}...')

throughput = {}
for bs in BATCH_SIZES:
    batch_queries = (valid_queries * ((bs // len(valid_queries)) + 1))[:bs]
    times = []
    for _ in range(20):
        t0    = time.perf_counter()
        ngram = vectorizer.transform(batch_queries)
        sym   = build_symbol_matrix(batch_queries)
        probs = rf_model.predict_proba(hstack([ngram, sym]))[:, 1]
        t1    = time.perf_counter()
        times.append(t1 - t0)
    throughput[bs] = bs / np.mean(times)

# MobileBERT throughput from NB17 cost analysis
MB_THROUGHPUT = {1: 33.1, 10: 315.0, 100: 1703.9}

print(f'\n{"─" * 65}')
print('THROUGHPUT (requests/second) — RF on CPU vs MobileBERT on GPU')
print(f'{"─" * 65}')
print(f'{"Batch Size":>12}  {"RF CPU":>12}  {"MobileBERT GPU":>16}  {"RF speedup":>10}')
print(f'{"─" * 65}')
for bs in BATCH_SIZES:
    mb = MB_THROUGHPUT.get(bs)
    mb_str     = f'{mb:>16.1f}' if mb else f'{"N/A":>16}'
    speedup_str = f'{throughput[bs]/mb:>10.1f}x' if mb else f'{"N/A":>10}'
    print(f'{bs:>12}  {throughput[bs]:>12.1f}  {mb_str}  {speedup_str}')
print(f'{"─" * 65}')


# ── AWS cost estimate — both batch=1 and batch=100 ────────────────
CPU_COST_PER_HR  = 0.170
GPU_COST_PER_HR  = 0.526
REQUESTS_PER_DAY = 1_000_000

def compute_cost(rps, cost_per_hr):
    hrs = (REQUESTS_PER_DAY / rps) / 3600
    return hrs, hrs * cost_per_hr

rf_b1_hrs,   rf_b1_cost   = compute_cost(throughput[1],   CPU_COST_PER_HR)
rf_b100_hrs, rf_b100_cost = compute_cost(throughput[100], CPU_COST_PER_HR)
mb_b1_hrs,   mb_b1_cost   = compute_cost(33.1,   GPU_COST_PER_HR)
mb_b100_hrs, mb_b100_cost = compute_cost(1703.9, GPU_COST_PER_HR)

print(f'\n{"─" * 80}')
print(f'AWS COST ESTIMATE — {REQUESTS_PER_DAY:,} requests/day')
print(f'{"─" * 80}')
print(f'{"Model":<28} {"Infra":>6} {"Batch":>6} {"RPS":>9} {"Hrs/day":>9} {"$/day":>8}')
print(f'{"─" * 80}')
print(f'{"RF (TF-IDF + Random Forest)":<28} {"CPU":>6} {"1":>6} {throughput[1]:>9.1f} {rf_b1_hrs:>9.4f} {rf_b1_cost:>8.4f}')
print(f'{"RF (TF-IDF + Random Forest)":<28} {"CPU":>6} {"100":>6} {throughput[100]:>9.1f} {rf_b100_hrs:>9.4f} {rf_b100_cost:>8.4f}')
print(f'{"MobileBERT zero-shot":<28} {"GPU":>6} {"1":>6} {33.1:>9.1f} {mb_b1_hrs:>9.2f} {mb_b1_cost:>8.4f}')
print(f'{"MobileBERT zero-shot":<28} {"GPU":>6} {"100":>6} {1703.9:>9.1f} {mb_b100_hrs:>9.4f} {mb_b100_cost:>8.4f}')
print(f'{"─" * 80}')
print(f'\nAt batch=1  : MobileBERT costs {mb_b1_cost/rf_b1_cost:.0f}x more per day than RF')
print(f'At batch=100: MobileBERT costs {mb_b100_cost/rf_b100_cost:.1f}x more per day than RF')
print(f'\nCPU pricing : c5.xlarge   ${CPU_COST_PER_HR}/hr (us-east-1 on-demand)')
print(f'GPU pricing : g4dn.xlarge ${GPU_COST_PER_HR}/hr (us-east-1 on-demand)')


# ── Save ──────────────────────────────────────────────────────────
cost_summary = {
    'test_queries':      len(valid_queries),
    'n_repeats_latency': N_REPEATS,
    'latency_ms_note':   'dominated by TF-IDF single-item transform overhead at batch=1',
    'latency_ms': {'rf': {
        'mean': round(float(np.mean(lat)), 4),
        'std':  round(float(np.std(lat)), 4),
        'p50':  round(float(np.percentile(lat, 50)), 4),
        'p95':  round(float(np.percentile(lat, 95)), 4),
        'p99':  round(float(np.percentile(lat, 99)), 4),
    }},
    'throughput_rps': {'rf': {str(bs): round(v, 1) for bs, v in throughput.items()}},
    'throughput_rps_mobilebert_gpu_ref': MB_THROUGHPUT,
    'aws_cost_per_day_usd': {
        'rf_cpu_batch1':          round(rf_b1_cost,   4),
        'rf_cpu_batch100':        round(rf_b100_cost,  4),
        'mobilebert_gpu_batch1':  round(mb_b1_cost,   4),
        'mobilebert_gpu_batch100': round(mb_b100_cost, 4),
    },
    'infrastructure': {'rf': 'CPU (c5.xlarge)', 'mobilebert': 'GPU (g4dn.xlarge)'},
    'cost_ratio_batch1':   round(mb_b1_cost/rf_b1_cost, 1),
    'cost_ratio_batch100': round(mb_b100_cost/rf_b100_cost, 1),
}

with open(f'{RESULTS_DIR}/18_rf_cost_analysis.json', 'w') as f:
    json.dump(cost_summary, f, indent=2)
print(f'\nSaved: {RESULTS_DIR}/18_rf_cost_analysis.json')


Parsed access log entries:
──────────────────────────────────────────────────────────────────────
[ 1] 1'OR'1'='1 Submit
[ 2] test'UNION SELECT username,password FROM users--
[ 3] 1'; waitfor delay '0:0:5';--
[ 4] admin'; DROP TABLE users;--
[ 5] shoes'AND 1=CAST((SELECT pg_sleep(5)) AS int)--
[ 6] admin'OR'a'='a
[ 7] 1 AND 1=1 UNION SELECT table_name FROM information_schema.tables--
[ 8] 1' OR '1'='1
[ 9] 1/**/OR/**/1=1--
[10] name'AND BENCHMARK(5000000,MD5(1))--

Successfully parsed: 10/10

Measuring single-request latency (100 repeats per query)...

─────────────────────────────────────────────────────────────────
SINGLE REQUEST LATENCY (ms) — CPU, 1 request at a time
─────────────────────────────────────────────────────────────────
Model                                   Mean      Std      P50      P95      P99
─────────────────────────────────────────────────────────────────
RF (TF-IDF + Random Forest)           23.774    0.722   23.674   24.866   26.281
──────────────────────────